# EU Corpus: Final Preprocessing Pipeline + Validation

**Pipeline:**
1. Load all documents
2. Build comprehensive stoplist
3. Preprocess text
4. Validate with top bigrams/trigrams (spot-check)
5. Score documents (reform vs criticism)
6. Export final dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
import warnings
warnings.filterwarnings('ignore')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

print('✓ Libraries ready')

## 1. Load Documents

In [ ]:
CORPUS_DIR = Path('scraped')
corpus_files = sorted(CORPUS_DIR.glob('*_corpus.txt'))

documents = []
for filepath in corpus_files:
    parts = filepath.stem.split('_')
    year = int(parts[-2])
    country = '_'.join(parts[:-2])
    structural_format = 'narrative' if year <= 2023 else 'chapter-based'
    
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    
    documents.append({
        'country': country,
        'year': year,
        'structural_format': structural_format,
        'raw_text': text
    })

df = pd.DataFrame(documents).sort_values(['country', 'year']).reset_index(drop=True)
print(f'Loaded {len(df)} documents')

## 2. Build Comprehensive Stoplist

In [ ]:
# Country names (from raw_eda.ipynb)
COUNTRY_NAMES = {
    'albania', 'kosovo', 'serbia', 'turkey', 'montenegro', 'bosnia', 'herzegovina', 'macedonia',
    'albanian', 'kosovar', 'serbian', 'montenegrin', 'bosnian', 'herzegovinian', 'macedonian', 'turkish',
    'türkiye', 'turk', 'serb', 'serbs',
    'pristina', 'prishtina', 'belgrade', 'beograd', 'podgorica', 'skopje', 'tirana', 'sarajevo', 'ankara', 'istanbul',
    'greece', 'croatia', 'russia', 'russian', 'ukraine', 'cyprus', 'syria'
}

# EU institutional terms
EU_PROCESS = {
    'eu', 'european', 'commission', 'council', 'parliament', 'union',
    'member', 'states', 'enlargement', 'accession', 'candidate',
    'negotiations', 'negotiation', 'chapter', 'chapters',
    'cluster', 'clusters', 'benchmark', 'benchmarks',
    'screening', 'intergovernmental', 'conference',
    'stabilisation', 'association', 'agreement', 'saa',
    'ipa', 'taiex', 'twinning', 'ipard',
    'acquis', 'commissioner',
    'opening', 'closing', 'closure', 'provisional',
    'conditionality', 'transposition', 'approximation', 'harmonisation',
    'directive', 'directives', 'regulation', 'regulations',
    'recommendation', 'recommendations',
    'framework', 'protocol', 'annex'
}

# EC boilerplate
EC_BOILERPLATE = {
    'report', 'reports', 'introduction', 'context',
    'summary', 'findings', 'implementation', 'progress',
    'alignment', 'continued', 'continues', 'remains', 'remain',
    'need', 'needs', 'ensure', 'adopted', 'provided',
    'overall', 'particular', 'areas', 'area', 'measures', 'measure',
    'efforts', 'effort', 'including', 'period', 'reporting',
    'government', 'governments', 'reform', 'reforms',
    'institutional', 'capacity', 'administrative', 'judiciary',
    'legislative', 'executive', 'authority', 'authorities',
    'page', 'final', 'document', 'staff', 'working',
    'com', 'swd', 'sw', 'ew', 'bw', 'iw', 'en', 'rem',
    'ibid', 'article', 'paragraph', 'graph',
    'accordance', 'pursuant', 'thereof',
    'country', 'countries', 'level', 'levels',
    'national', 'international', 'regional', 'local',
    'public', 'civil', 'society', 'state',
    'appoint', 'appointed', 'appointment',
    'largely', 'key', 'opinion', 'audio'
}

# WB institutions
WB_INSTITUTIONS = {
    'hjpc', 'hjc', 'hpc', 'aca', 'spak', 'boa', 'instat',
    'osce', 'echr', 'tca', 'csos', 'cso', 'ipard',
    'rem', 'ratel', 'nis', 'par', 'wbif', 'seeto',
    'magistrates', 'magistrate', 'inspectorate', 'inspectorates',
    'ombudsman', 'prosecutor', 'prosecutors', 'prosecutorial'
}

# Dates and noise
YEARS_SET = {str(y) for y in range(2000, 2030)}
MONTHS = {'january','february','march','april','may','june','july','august','september','october','november','december','jan','feb','mar','apr','jun','jul','aug','sep','oct','nov','dec'}
NOISE = {'pandemic', 'covid', 'coronavirus', 'earthquakes', 'earthquake', 'military', 'presidential', 'eastern'}

# Combine
english_stops = set(stopwords.words('english'))
STOPWORDS = list(english_stops | COUNTRY_NAMES | EU_PROCESS | EC_BOILERPLATE | WB_INSTITUTIONS | YEARS_SET | MONTHS | NOISE)

print(f'✓ Stoplist: {len(STOPWORDS)} terms')

## 3. Preprocess Text

In [ ]:
def preprocess(text, stoplist):
    text = re.sub(r'\\d+', '', text)  # Remove numbers
    text = text.lower()
    text = re.sub(r'[^\\w\\s-]', ' ', text)  # Remove punctuation, keep hyphens
    tokens = text.split()
    tokens = [t for t in tokens if t and t not in stoplist and len(t) > 1]
    return tokens

df['cleaned_tokens'] = df['raw_text'].apply(lambda x: preprocess(x, STOPWORDS))
df['cleaned_total_tokens'] = df['cleaned_tokens'].apply(len)

raw_total = df['raw_text'].apply(lambda x: len(x.split())).sum()
cleaned_total = df['cleaned_total_tokens'].sum()
reduction = (1 - cleaned_total/raw_total)*100

print(f'✓ Preprocessing complete')
print(f'  Raw tokens: {raw_total:,}')
print(f'  Cleaned tokens: {cleaned_total:,}')
print(f'  Reduction: {reduction:.1f}%')

## 4. Validate: Top Bigrams and Trigrams

In [ ]:
# Extract all tokens
all_tokens = []
for tokens in df['cleaned_tokens']:
    all_tokens.extend(tokens)

# Top unigrams
unigram_counts = Counter(all_tokens)
top_unigrams = unigram_counts.most_common(25)

print('\\n' + '='*70)
print('Top 25 Unigrams (Validation Check)')
print('='*70)
print(f"{'Rank':<5} {'Token':<35} {'Count':>10}")
print('-'*50)
for i, (token, count) in enumerate(top_unigrams, 1):
    print(f'{i:<5} {token:<35} {count:>10}')

In [ ]:
# Extract bigrams
bigrams = []
for tokens in df['cleaned_tokens']:
    for i in range(len(tokens) - 1):
        bigrams.append((tokens[i], tokens[i+1]))

bigram_counts = Counter(bigrams)
top_bigrams = bigram_counts.most_common(25)

print('\\n' + '='*80)
print('Top 25 Bigrams (Validation Check)')
print('='*80)
print(f"{'Rank':<5} {'Bigram':<60} {'Count':>10}")
print('-'*75)
for i, (bigram, count) in enumerate(top_bigrams, 1):
    bigram_str = f'{bigram[0]} {bigram[1]}'
    print(f'{i:<5} {bigram_str:<60} {count:>10}')

In [ ]:
# Extract trigrams
trigrams = []
for tokens in df['cleaned_tokens']:
    for i in range(len(tokens) - 2):
        trigrams.append((tokens[i], tokens[i+1], tokens[i+2]))

trigram_counts = Counter(trigrams)
top_trigrams = trigram_counts.most_common(20)

print('\\n' + '='*90)
print('Top 20 Trigrams (Validation Check)')
print('='*90)
print(f"{'Rank':<5} {'Trigram':<70} {'Count':>10}")
print('-'*85)
for i, (trigram, count) in enumerate(top_trigrams, 1):
    trigram_str = f'{trigram[0]} {trigram[1]} {trigram[2]}'
    print(f'{i:<5} {trigram_str:<70} {count:>10}')

print('\\n✓ Spot Check:')
print('  - Are country names removed? (Should NOT see kosovo, serbia, albania, etc.)')
print('  - Are EU terms removed? (Should NOT see commission, council, parliament, etc.)')
print('  - Do you see substantive policy content? (governance, reform, judiciary, etc.)')

## 5. Score Documents

In [ ]:
reform_words = {'progress', 'progressed', 'advanced', 'advances', 'adopted', 'adopting', 'implemented', 'implementing', 'strengthened', 'strengthens', 'strengthening', 'improved', 'improving', 'improvement', 'established', 'establishing', 'achieved', 'achieves', 'achievement', 'accelerated', 'accelerate', 'completed', 'completion', 'completing', 'good', 'better', 'well', 'success', 'successful', 'successfully', 'effective', 'effectively', 'positive', 'positively', 'enhance', 'enhanced', 'enhancement', 'functional', 'functioning', 'efficient', 'efficiency'}

criticism_words = {'concern', 'concerns', 'concerning', 'weak', 'weaken', 'weakness', 'lack', 'lacked', 'lacking', 'delayed', 'delay', 'delays', 'limited', 'limiting', 'incomplete', 'incompleteness', 'decreased', 'decrease', 'declining', 'decline', 'insufficient', 'insufficiently', 'challenge', 'challenges', 'challenging', 'persisting', 'persist', 'persistent', 'backsliding', 'backslide', 'difficult', 'difficulty', 'fail', 'failed', 'failure', 'failing', 'problem', 'problems', 'problematic', 'issue', 'issues', 'risk', 'risks', 'serious', 'seriously', 'need', 'needed', 'required', 'require', 'worrying', 'worried', 'inadequate', 'undermining', 'undermine'}

def score(tokens):
    total = len(tokens)
    reform_count = sum(1 for t in tokens if t in reform_words)
    crit_count = sum(1 for t in tokens if t in criticism_words)
    reform_pct = (reform_count / total * 100) if total > 0 else 0
    crit_pct = (crit_count / total * 100) if total > 0 else 0
    return reform_count, crit_count, reform_pct, crit_pct

results = df['cleaned_tokens'].apply(lambda x: pd.Series(score(x)))
results.columns = ['reform_count', 'criticism_count', 'reform_score', 'criticism_score']
df = pd.concat([df, results], axis=1)
df['net_score'] = df['reform_score'] - df['criticism_score']

print('\\n' + '='*80)
print('Scores (on cleaned tokens)')
print('='*80)
print(f"{'Country':<25} {'Year':>5} {'Format':<15} {'Reform%':>8} {'Crit%':>8} {'Net':>8}")
print('-'*80)
for _, row in df.iterrows():
    country = row['country'].replace('_', ' ').title()
    print(f'{country:<25} {row["year"]:>5} {row["structural_format"]:<15} {row["reform_score"]:>7.2f}% {row["criticism_score"]:>7.2f}% {row["net_score"]:>7.3f}')

print(f"\\n{'Mean':<25} {'':>5} {'':>15} {df['reform_score'].mean():>7.2f}% {df['criticism_score'].mean():>7.2f}% {df['net_score'].mean():>7.3f}")

## 6. Export Final Dataset

In [ ]:
export_df = df[[
    'country', 'year', 'structural_format',
    'cleaned_total_tokens',
    'reform_count', 'criticism_count',
    'reform_score', 'criticism_score', 'net_score'
]].copy()

export_df.to_csv('EU_corpus_final.csv', index=False)

print('\\n' + '='*80)
print('✓ EXPORT COMPLETE: EU_corpus_final.csv')
print('='*80)
print(f'N={len(export_df)} observations')
print(f'\\nReady for analysis!')